In [10]:
# Sentinel-1 EW模式图像匹配与可视化系统
# 适用于Jupyter Notebook环境

import ee
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import logging
import time
import re
import folium
import matplotlib.pyplot as plt
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')
# 设置日志
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

In [16]:

def initialize_gee():
    """初始化Google Earth Engine"""
    try:
        ee.Initialize()
        print("✓ GEE 初始化成功!")
        logger.info("Earth Engine 初始化成功")
        return True
    except Exception as e:
        print(f"✗ GEE 初始化失败: {e}")
        print("请先运行: ee.Authenticate()")
        logger.error(f"Earth Engine 初始化失败: {e}")
        return False

class Config:
    """配置类"""
    def __init__(self):
        self.TIME_WINDOW_HOURS = 7      # 时间窗口
        self.CLOUD_THRESHOLD = 20       # 云量阈值
        self.MIN_OVERLAP_RATIO = 0.5    # 最小重叠率
        self.DELAY_SECONDS = 0.5        # API调用延迟
        self.MAX_IMAGES_PER_COLLECTION = 20  # 每个集合最大图像数
        
    def update_config(self, **kwargs):
        """更新配置参数"""
        for key, value in kwargs.items():
            if hasattr(self, key.upper()):
                setattr(self, key.upper(), value)
                print(f"✓ 更新配置 {key}: {value}")
            else:
                print(f"⚠ 未知配置参数: {key}")
                
CSV_PATH = "projects/vaulted-hawk-469808-i5/assets/time_match_2023"

In [4]:
class CSVHandler:
    """CSV数据处理类"""
    
    def __init__(self, min_matched_points=60):
        self.min_matched_points = min_matched_points
        self.df = None
        self.ew_filtered_df = None
    
    def load_csv_file(self, csv_file_path, scene_name_column='sceneName'):
        """
        从本地CSV文件加载数据，自动筛选EW模式
        
        Args:
            csv_file_path: 本地CSV文件路径
            scene_name_column: 场景名称列的名称
            
        Returns:
            pd.DataFrame: 筛选后的EW模式数据
        """
        try:
            # 加载CSV文件
            self.df = pd.read_csv(csv_file_path)
            print(f"✓ 成功加载CSV文件: {len(self.df)} 条记录")
            
            # 检查必要的列
            if scene_name_column not in self.df.columns:
                raise ValueError(f"CSV中未找到列：{scene_name_column}")
            
            # 显示列信息
            print(f"CSV列名: {list(self.df.columns)}")
            
            # 筛选EW模式数据
            self.ew_filtered_df = self.filter_ew_mode_data(scene_name_column)
            
            return self.ew_filtered_df
            
        except Exception as e:
            logger.error(f"加载CSV文件失败: {e}")
            raise
    
    def is_ew_mode(self, scene_name):
        """
        判断场景名称是否为EW模式
        
        Args:
            scene_name: 场景名称
            
        Returns:
            bool: 是否为EW模式
        """
        if pd.isna(scene_name):
            return False
        
        scene_str = str(scene_name)
        # Sentinel-1场景名称格式检查EW模式
        # 格式: S1A_EW_GRDM_1SDH_20230101T120000_20230101T120025_046123_058456_1234
        return '_EW_' in scene_str
    
    def filter_ew_mode_data(self, scene_name_column):
        """
        筛选EW模式数据
        
        Args:
            scene_name_column: 场景名称列名
            
        Returns:
            pd.DataFrame: 筛选后的数据
        """
        if self.df is None:
            raise ValueError("请先加载CSV数据")
        
        print("筛选EW模式的S1数据...")
        
        # 筛选条件
        ew_mask = self.df[scene_name_column].apply(self.is_ew_mode)
        
        # 如果有num_matched_points列，也进行筛选
        if 'num_matched_points' in self.df.columns:
            points_mask = self.df['num_matched_points'] > self.min_matched_points
            combined_mask = ew_mask & points_mask
            print(f"应用额外筛选条件: num_matched_points > {self.min_matched_points}")
        else:
            combined_mask = ew_mask
            print("未找到num_matched_points列，仅按EW模式筛选")
        
        filtered_df = self.df[combined_mask].copy()
        
        print(f"✓ 筛选完成:")
        print(f"  原始数据: {len(self.df)} 条")
        print(f"  EW模式数据: {combined_mask.sum()} 条")
        print(f"  最终筛选结果: {len(filtered_df)} 条")
        
        return filtered_df
    
    def get_row_data(self, row_index, scene_name_column='sceneName'):
        """
        获取指定行的数据
        
        Args:
            row_index: 行索引
            scene_name_column: 场景名称列名
            
        Returns:
            dict: 行数据信息
        """
        if self.ew_filtered_df is None:
            raise ValueError("请先加载并筛选CSV数据")
        
        if row_index >= len(self.ew_filtered_df):
            raise IndexError(f"行索引 {row_index} 超出范围 (最大: {len(self.ew_filtered_df)-1})")
        
        # 获取行数据
        row_data = self.ew_filtered_df.iloc[row_index]
        scene_name = row_data[scene_name_column]
        
        print(f"\n=== 第 {row_index} 行数据 ===")
        print(f"Scene Name: {scene_name}")
        
        # 显示其他可用信息
        for col, value in row_data.items():
            if col != scene_name_column and not pd.isna(value):
                print(f"{col}: {value}")
        
        result = {
            'row_index': row_index,
            'scene_name': scene_name,
            'raw_data': row_data.to_dict(),
            'is_ew_mode': self.is_ew_mode(scene_name)
        }
        
        return result
    
    def show_data_preview(self, n_rows=5, scene_name_column='sceneName'):
        """
        显示数据预览
        
        Args:
            n_rows: 显示行数
            scene_name_column: 场景名称列名
        """
        if self.ew_filtered_df is None:
            print("请先加载CSV数据")
            return
        
        print(f"\n=== EW模式数据预览 (前 {n_rows} 行) ===")
        
        preview_df = self.ew_filtered_df.head(n_rows)
        
        for idx, (_, row) in enumerate(preview_df.iterrows()):
            print(f"\n第 {idx} 行:")
            print(f"  Scene Name: {row[scene_name_column]}")
            
            # 显示其他重要列
            important_cols = ['num_matched_points', 'Mode', 'productType', 'platformHeading']
            for col in important_cols:
                if col in row and not pd.isna(row[col]):
                    print(f"  {col}: {row[col]}")
        
        print(f"\n数据形状: {self.ew_filtered_df.shape}")
        print(f"可用列: {list(self.ew_filtered_df.columns)}")

In [ ]:
import ee
import re
from datetime import datetime, timedelta
import logging

class ImageMatcher:
    """图像匹配器类"""
    
    def __init__(self, config):
        self.config = config
        self.current_result = None
    
    def extract_date_from_scene_name(self, scene_name):
        """
        从Sentinel-1 EW模式 scene name中提取日期
        
        Args:
            scene_name: Sentinel-1场景名称
            
        Returns:
            datetime: 解析出的日期时间
        """
        try:
            # EW模式 scene name示例:
            # S1A_EW_GRDM_1SDH_20210101T123456_20210101T123520_036123_043B2A_1234
            
            # 查找时间戳模式
            datetime_pattern = r'(\d{8}T\d{6})'
            match = re.search(datetime_pattern, scene_name)
            
            if match:
                date_str = match.group(1)
                parsed_date = datetime.strptime(date_str, '%Y%m%dT%H%M%S')
                print(f"✓ 从scene name提取日期: {parsed_date.strftime('%Y-%m-%d %H:%M:%S')}")
                return parsed_date
            
            # 如果没找到完整时间戳，尝试只提取日期
            date_pattern = r'(\d{8})'
            match = re.search(date_pattern, scene_name)
            if match:
                date_str = match.group(1)
                parsed_date = datetime.strptime(date_str, '%Y%m%d')
                print(f"✓ 从scene name提取日期(仅日期): {parsed_date.strftime('%Y-%m-%d')}")
                return parsed_date
                
            print(f"⚠ 无法从scene name中提取日期: {scene_name}")
            return None
            
        except Exception as e:
            logger.error(f"日期提取失败: {e}")
            return None
    
    def find_sentinel1_image(self, scene_name, roi=None):
        """
        查找Sentinel-1影像（已知为EW模式）
        
        Args:
            scene_name: 场景名称（已筛选为EW模式）
            roi: 感兴趣区域
            
        Returns:
            dict: Sentinel-1影像信息
        """
        try:
            # 构建Sentinel-1集合，直接查找EW模式
            s1_collection = ee.ImageCollection('COPERNICUS/S1_GRD') \
                .filter(ee.Filter.eq('instrumentMode', 'EW'))
            
            # 尝试不同的筛选方式
            filters = [
                ee.Filter.eq('system:index', scene_name),
                ee.Filter.stringContains('system:id', scene_name),
                ee.Filter.stringContains('PRODUCT_ID', scene_name)
            ]
            
            s1_image = None
            for filter_condition in filters:
                temp_collection = s1_collection.filter(filter_condition)
                
                if roi:
                    temp_collection = temp_collection.filterBounds(roi)
                
                if temp_collection.size().getInfo() > 0:
                    s1_image = temp_collection.first()
                    break
            
            if s1_image is None:
                return {'error': f'未找到场景名称为 {scene_name} 的Sentinel-1影像'}
            
            # 获取影像信息
            image_info = s1_image.getInfo()
            acquisition_time = ee.Date(s1_image.get('system:time_start'))
            
            return {
                'image': s1_image,
                'image_id': image_info['id'],
                'acquisition_time': acquisition_time,
                'acquisition_time_str': acquisition_time.format('YYYY-MM-dd HH:mm:ss').getInfo(),
                'properties': image_info.get('properties', {}),
                'geometry': s1_image.geometry()
            }
            
        except Exception as e:
            return {'error': f'处理场景 {scene_name} 时出错: {str(e)}'}
    
    def find_matching_images(self, s1_info, sensor_type='sentinel2'):
        """
        查找时间窗口内的匹配影像并进行重合率检查
        
        Args:
            s1_info: Sentinel-1影像信息
            sensor_type: 'sentinel2' 或 'landsat8'
            
        Returns:
            list: 通过重合率检查的匹配影像列表
        """
        if 'error' in s1_info:
            return []
        
        try:
            acquisition_time = s1_info['acquisition_time']
            s1_geometry = s1_info['geometry']
            
            # 计算时间窗口
            time_start = acquisition_time.advance(-self.time_window_hours, 'hour')
            time_end = acquisition_time.advance(self.time_window_hours, 'hour')
            
            if sensor_type == 'sentinel2':
                collection = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
                    .filterDate(time_start, time_end) \
                    .filterBounds(s1_geometry) \
                    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', self.cloud_threshold))
                    
            elif sensor_type == 'landsat8':
                collection = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2') \
                    .filterDate(time_start, time_end) \
                    .filterBounds(s1_geometry) \
                    .filter(ee.Filter.lt('CLOUD_COVER', self.cloud_threshold))
            else:
                return []
            
            collection_size = collection.size().getInfo()
            
            if collection_size == 0:
                return []
            
            # 限制返回数量以避免内存问题
            max_images = min(collection_size, 10)
            image_list = collection.limit(max_images).getInfo()
            
            matched_images = []
            overlap_stats = {'checked': 0, 'passed': 0, 'failed': 0}
            
            for img_info in image_list['features']:
                overlap_stats['checked'] += 1
                
                try:
                    # 获取匹配影像的几何
                    matched_image = ee.Image(img_info['id'])
                    matched_geometry = matched_image.geometry()
                    
                    # 进行重合率检查
                    overlap_check = self.check_overlap_quality(s1_geometry, matched_geometry)
                    
                    if overlap_check['passes_check']:
                        overlap_stats['passed'] += 1
                        
                        img_time = ee.Date(img_info['properties']['system:time_start'])
                        time_diff = img_time.difference(acquisition_time, 'hour').getInfo()
                        
                        matched_images.append({
                            'image_id': img_info['id'],
                            'acquisition_time': img_time.format('YYYY-MM-dd HH:mm:ss').getInfo(),
                            'time_difference_hours': round(time_diff, 2),
                            'properties': img_info['properties'],
                            'sensor_type': sensor_type,
                            'overlap_ratio': overlap_check['overlap_ratio'],
                            'overlap_percentage': overlap_check['overlap_percentage']
                        })
                    else:
                        overlap_stats['failed'] += 1
                        logger.debug(f"影像 {img_info['id']} 重合率过低: {overlap_check['overlap_percentage']}%")
                        
                except Exception as e:
                    logger.warning(f"处理影像 {img_info['id']} 时出错: {e}")
                    continue
            
            if overlap_stats['checked'] > 0:
                logger.info(f"{sensor_type} 重合率检查: {overlap_stats['checked']}个检查, {overlap_stats['passed']}个通过, {overlap_stats['failed']}个未通过")
            
            return matched_images
            
        except Exception as e:
            logger.error(f"查找匹配影像时出错: {e}")
            return []
        
        
    def calculate_overlap_ratio(self, geom1, geom2):
        """
        计算两个几何对象的重叠率
        
        Args:
            geom1: 第一个几何对象（ee.Geometry）
            geom2: 第二个几何对象（ee.Geometry）
            
        Returns:
            float: 重叠率（0-1之间）
        """
        try:
            # 计算相交区域
            intersection = geom1.intersection(geom2, maxError=1000)
            
            # 计算面积
            intersection_area = intersection.area(maxError=1000).getInfo()
            geom1_area = geom1.area(maxError=1000).getInfo()
            geom2_area = geom2.area(maxError=1000).getInfo()
            
            # 避免除零错误
            if geom1_area == 0 or geom2_area == 0:
                return 0.0
            
            # 计算重叠率（相对于较小影像的比例）
            min_area = min(geom1_area, geom2_area)
            overlap_ratio = intersection_area / min_area
            
            # 确保在0-1范围内
            return min(max(overlap_ratio, 0.0), 1.0)
            
        except Exception as e:
            logger.warning(f"计算重叠率时出错: {e}")
            return 0.0
    
    def process_single_scene(self, scene_name, roi=None):
        """
        处理单个场景
        
        Args:
            scene_name: 场景名称
            roi: 感兴趣区域
            
        Returns:
            list: 处理结果列表
        """
        scene_results = []
        
        # 查找Sentinel-1影像
        s1_info = self.find_sentinel1_image(scene_name, roi)
        
        if 'error' in s1_info:
            scene_results.append({
                's1_scene_name': scene_name,
                'error': s1_info['error'],
                'status': 'failed'
            })
            return scene_results
        
        # 查找Sentinel-2匹配
        s2_matches = self.find_matching_images(s1_info, 'sentinel2')
        
        # 查找Landsat-8匹配
        l8_matches = self.find_matching_images(s1_info, 'landsat8')
        
        # 记录Sentinel-2匹配结果
        if s2_matches:
            for s2_match in s2_matches:
                scene_results.append({
                    's1_scene_name': scene_name,
                    's1_image_id': s1_info['image_id'],
                    's1_acquisition_time': s1_info['acquisition_time_str'],
                    'matched_image_id': s2_match['image_id'],
                    'matched_acquisition_time': s2_match['acquisition_time'],
                    'time_difference_hours': s2_match['time_difference_hours'],
                    'overlap_ratio': s2_match['overlap_ratio'],
                    'overlap_percentage': s2_match['overlap_percentage'],
                    'sensor_pair': 'Sentinel-1 & Sentinel-2',
                    'matched_sensor': 'Sentinel-2',
                    'status': 'success'
                })
        
        # 记录Landsat-8匹配结果
        if l8_matches:
            for l8_match in l8_matches:
                scene_results.append({
                    's1_scene_name': scene_name,
                    's1_image_id': s1_info['image_id'],
                    's1_acquisition_time': s1_info['acquisition_time_str'],
                    'matched_image_id': l8_match['image_id'],
                    'matched_acquisition_time': l8_match['acquisition_time'],
                    'time_difference_hours': l8_match['time_difference_hours'],
                    'overlap_ratio': l8_match['overlap_ratio'],
                    'overlap_percentage': l8_match['overlap_percentage'],
                    'sensor_pair': 'Sentinel-1 & Landsat-8',
                    'matched_sensor': 'Landsat-8',
                    'status': 'success'
                })
        
        # 如果没有找到任何匹配
        if not s2_matches and not l8_matches:
            scene_results.append({
                's1_scene_name': scene_name,
                's1_image_id': s1_info['image_id'],
                's1_acquisition_time': s1_info['acquisition_time_str'],
                'matched_image_id': 'No match found',
                'sensor_pair': 'Sentinel-1 only',
                'status': 'no_match'
            })
        
        return scene_results
    


In [13]:
# ==================== 可视化类 ====================
class ImageVisualizer:
    """图像可视化器类"""
    
    def __init__(self):
        self.current_map = None
        self.current_result = None
    
    def create_base_map(self, geometry, zoom_start=10):
        """创建基础地图"""
        try:
            # 获取几何中心点
            centroid = geometry.centroid().coordinates().getInfo()
            lon, lat = centroid[0], centroid[1]
            
            print(f"地图中心点: 纬度 {lat:.4f}, 经度 {lon:.4f}")
            
            # 创建地图
            m = folium.Map(
                location=[lat, lon], 
                zoom_start=zoom_start,
                tiles='OpenStreetMap'
            )
            
            # 添加其他底图选项
            folium.TileLayer('Stamen Terrain').add_to(m)
            folium.TileLayer('CartoDB positron').add_to(m)
            
            # 获取边界框
            bounds = geometry.bounds().getInfo()['coordinates'][0]
            # 转换为folium需要的格式 [[south, west], [north, east]]
            sw = [bounds[0][1], bounds[0][0]]  # [lat, lon]
            ne = [bounds[2][1], bounds[2][0]]  # [lat, lon]
            
            # 添加AOI边界
            folium.Rectangle(
                bounds=[sw, ne],
                color='red',
                weight=2,
                fill=False,
                popup='感兴趣区域'
            ).add_to(m)
            
            return m, (sw, ne)
            
        except Exception as e:
            logger.error(f"创建基础地图失败: {e}")
            return None, None
    
    def add_sentinel1_layer(self, map_obj, s1_info, bounds):
        """添加Sentinel-1图层"""
        try:
            if 'error' in s1_info:
                print("⚠ Sentinel-1图像有错误，跳过可视化")
                return False
            
            s1_image = s1_info['image']
            
            # SAR可视化参数
            vis_params = {
                'min': -25,
                'max': 5,
                'bands': ['VV'],
                'dimensions': 512,
                'format': 'png'
            }
            
            # 获取缩略图URL
            try:
                thumbnail_url = s1_image.getThumbURL(vis_params)
                
                # 添加图层
                folium.raster_layers.ImageOverlay(
                    image=thumbnail_url,
                    bounds=bounds,
                    name=f'Sentinel-1 SAR (EW模式)',
                    opacity=0.8
                ).add_to(map_obj)
                
                print("✓ Sentinel-1图层已添加")
                return True
                
            except Exception as e:
                print(f"⚠ 获取Sentinel-1缩略图失败: {e}")
                return False
                
        except Exception as e:
            logger.error(f"添加Sentinel-1图层失败: {e}")
            return False
    
    def add_optical_layer(self, map_obj, optical_match, bounds, sensor_type):
        """添加光学图像图层"""
        try:
            if not optical_match:
                print(f"⚠ 没有{sensor_type}匹配图像，跳过")
                return False
            
            # 选择最佳匹配
            best_match = optical_match[0] if isinstance(optical_match, list) else optical_match
            optical_image = best_match['image']
            
            # 根据传感器类型设置可视化参数
            if sensor_type == 'sentinel2':
                vis_params = {
                    'min': 0,
                    'max': 3000,
                    'bands': ['B4', 'B3', 'B2'],  # RGB
                    'dimensions': 512,
                    'format': 'png'
                }
                layer_name = f"Sentinel-2 RGB (重叠率: {best_match['overlap_percentage']:.1f}%)"
                
            elif sensor_type == 'landsat8':
                vis_params = {
                    'min': 0,
                    'max': 30000,
                    'bands': ['SR_B4', 'SR_B3', 'SR_B2'],  # RGB
                    'dimensions': 512,
                    'format': 'png'
                }
                layer_name = f"Landsat-8 RGB (重叠率: {best_match['overlap_percentage']:.1f}%)"
            else:
                return False
            
            # 获取缩略图URL
            try:
                thumbnail_url = optical_image.getThumbURL(vis_params)
                
                # 添加图层
                folium.raster_layers.ImageOverlay(
                    image=thumbnail_url,
                    bounds=bounds,
                    name=layer_name,
                    opacity=0.7
                ).add_to(map_obj)
                
                print(f"✓ {sensor_type}图层已添加")
                return True
                
            except Exception as e:
                print(f"⚠ 获取{sensor_type}缩略图失败: {e}")
                return False
                
        except Exception as e:
            logger.error(f"添加{sensor_type}图层失败: {e}")
            return False
    
    def visualize_matching_result(self, result, zoom_start=10):
        """可视化匹配结果"""
        if result is None or 'error' in result:
            print("✗ 无法可视化，结果包含错误")
            return None
        
        print(f"\n=== 创建可视化地图 ===")
        print(f"场景: {result['scene_name']}")
        
        try:
            # 获取Sentinel-1几何体
            s1_info = result['sentinel1_info']
            geometry = s1_info['geometry']
            
            # 创建基础地图
            m, bounds = self.create_base_map(geometry, zoom_start)
            if m is None:
                return None
            
            # 添加Sentinel-1图层
            self.add_sentinel1_layer(m, s1_info, bounds)
            
            # 添加Sentinel-2图层
            if result['sentinel2_matches']:
                self.add_optical_layer(m, result['sentinel2_matches'], bounds, 'sentinel2')
            
            # 添加Landsat-8图层
            if result['landsat8_matches']:
                self.add_optical_layer(m, result['landsat8_matches'], bounds, 'landsat8')
            
            # 添加图层控制
            folium.LayerControl().add_to(m)
            
            # 添加场景信息
            scene_info = f"""
            <b>场景信息:</b><br>
            Scene Name: {result['scene_name']}<br>
            Sentinel-1 ID: {s1_info['image_id']}<br>
            获取时间: {s1_info['actual_time'].strftime('%Y-%m-%d %H:%M:%S')}<br>
            Sentinel-2匹配: {result['summary']['s2_count']} 个<br>
            Landsat-8匹配: {result['summary']['l8_count']} 个
            """
            
            folium.Marker(
                [bounds[0][0] + (bounds[1][0] - bounds[0][0]) * 0.1, 
                 bounds[0][1] + (bounds[1][1] - bounds[0][1]) * 0.1],
                popup=scene_info,
                icon=folium.Icon(color='blue', icon='info-sign')
            ).add_to(m)
            
            self.current_map = m
            self.current_result = result
            
            print("✓ 可视化地图创建成功")
            return m
            
        except Exception as e:
            logger.error(f"可视化失败: {e}")
            return None
    
def create_summary_plot(self, result):
        """创建匹配摘要图表"""
        if result is None or 'error' in result:
            print("无法创建摘要图表")
            return None
        
        fig, axes = plt.subplots(2, 2, figsize=(12, 8))
        fig.suptitle(f"匹配结果摘要: {result['scene_name']}", fontsize=14)
        
        # 匹配数量统计
        ax1 = axes[0, 0]
        sensors = ['Sentinel-2', 'Landsat-8']
        counts = [result['summary']['s2_count'], result['summary']['l8_count']]
        ax1.bar(sensors, counts, color=['blue', 'green'])
        ax1.set_title('匹配图像数量')
        ax1.set_ylabel('数量')
        
        # 时间差分布
        ax2 = axes[0, 1]
        time_diffs = []
        labels = []
        
        for match in result['sentinel2_matches']:
            time_diffs.append(match['time_difference_hours'])
            labels.append('S2')
        
        for match in result['landsat8_matches']:
            time_diffs.append(match['time_difference_hours'])
            labels.append('L8')
        
        if time_diffs:
            colors = ['blue' if l == 'S2' else 'green' for l in labels]
            ax2.scatter(range(len(time_diffs)), time_diffs, c=colors, alpha=0.7)
            ax2.set_title('时间差分布')
            ax2.set_xlabel('匹配图像索引')
            ax2.set_ylabel('时间差 (小时)')
        else:
            ax2.text(0.5, 0.5, '无匹配图像', ha='center', va='center', transform=ax2.transAxes)
        
        # 重叠率分布
        ax3 = axes[1, 0]
        overlap_ratios = []
        overlap_labels = []
        
        for match in result['sentinel2_matches']:
            overlap_ratios.append(match['overlap_percentage'])
            overlap_labels.append('S2')
        
        for match in result['landsat8_matches']:
            overlap_ratios.append(match['overlap_percentage'])
            overlap_labels.append('L8')
        
        if overlap_ratios:
            colors = ['blue' if l == 'S2' else 'green' for l in overlap_labels]
            ax3.scatter(range(len(overlap_ratios)), overlap_ratios, c=colors, alpha=0.7)
            ax3.set_title('重叠率分布')
            ax3.set_xlabel('匹配图像索引')
            ax3.set_ylabel('重叠率 (%)')
        else:
            ax3.text(0.5, 0.5, '无匹配图像', ha='center', va='center', transform=ax3.transAxes)
        
        # 基本信息
        ax4 = axes[1, 1]
        ax4.axis('off')
        info_text = f"""
        基本信息:
        场景名称: {result['scene_name'][:30]}...
        行索引: {result['row_index']}
        Sentinel-1获取时间: {result['sentinel1_info']['actual_time'].strftime('%Y-%m-%d %H:%M')}
        总匹配数: {result['summary']['total_matches']}
        处理状态: {result['status']}
        """
        ax4.text(0.1, 0.9, info_text, transform=ax4.transAxes, 
                fontsize=10, verticalalignment='top', fontfamily='monospace')
        
        plt.tight_layout()
        return fig

In [14]:
# 完整的工作流程类
class VisualizationWorkflow:
    """完整的可视化工作流程"""
    
    def __init__(self, csv_file_path, scene_name_column='sceneName', min_matched_points=60):
        self.csv_handler = CSVHandler(min_matched_points)
        self.config = Config()
        self.matcher = ImageMatcher(self.config)
        self.visualizer = ImageVisualizer()
        
        # 加载数据
        self.filtered_data = self.csv_handler.load_csv_file(csv_file_path, scene_name_column)
        self.scene_name_column = scene_name_column
        self.current_index = 0
        
    def process_and_visualize_row(self, row_index, roi=None, zoom_start=10):
        """处理并可视化指定行"""
        if row_index >= len(self.filtered_data):
            print(f"行索引 {row_index} 超出范围 (最大: {len(self.filtered_data)-1})")
            return None, None, None
        
        # 获取行数据
        row_data = self.csv_handler.get_row_data(row_index, self.scene_name_column)
        
        # 执行匹配
        match_result = self.matcher.match_single_row(row_data, roi)
        
        if match_result['status'] == 'failed':
            print(f"处理失败: {match_result['error']}")
            return row_data, match_result, None
        
        # 创建可视化
        map_obj = self.visualizer.visualize_matching_result(match_result, zoom_start)

In [8]:
# success = initialize_gee()
# 创建CSV处理器
csv_handler = CSVHandler(min_matched_points=60)
csv_file_path = r"E:\NWP\CS2_S1_matched\time_match_2023.csv"
scene_name_column = 'sceneName'
# 加载和筛选数据
filtered_data = csv_handler.load_csv_file(csv_file_path, scene_name_column)

if filtered_data is not None and len(filtered_data) > 0:
    # 显示数据预览
    csv_handler.show_data_preview(3, scene_name_column)
    
    # 测试获取特定行
    print(f"\n=== 测试获取第0行数据 ===")
    row_data = csv_handler.get_row_data(0, scene_name_column)
    

else:
    print("✗ 没有找到符合条件的EW模式数据")


✓ 成功加载CSV文件: 360 条记录
CSV列名: ['sceneName', 's1_start', 's1_end', 's1_url', 'cs2_path', 'cs2_time_min', 'cs2_time_max', 'num_matched_points']
筛选EW模式的S1数据...
应用额外筛选条件: num_matched_points > 60
✓ 筛选完成:
  原始数据: 360 条
  EW模式数据: 61 条
  最终筛选结果: 61 条

=== EW模式数据预览 (前 3 行) ===

第 0 行:
  Scene Name: S1A_EW_GRDM_1SDH_20230630T134749_20230630T134853_049216_05EB02_5C83
  num_matched_points: 89

第 1 行:
  Scene Name: S1A_EW_GRDM_1SDH_20230630T120922_20230630T121027_049215_05EAFB_1952
  num_matched_points: 71

第 2 行:
  Scene Name: S1A_EW_GRDM_1SDH_20230629T144524_20230629T144629_049202_05EA9F_5AF6
  num_matched_points: 73

数据形状: (61, 8)
可用列: ['sceneName', 's1_start', 's1_end', 's1_url', 'cs2_path', 'cs2_time_min', 'cs2_time_max', 'num_matched_points']

=== 测试获取第0行数据 ===

=== 第 0 行数据 ===
Scene Name: S1A_EW_GRDM_1SDH_20230630T134749_20230630T134853_049216_05EB02_5C83
s1_start: 2023-06-30 13:47:49
s1_end: 2023-06-30 13:48:53
s1_url: https://datapool.asf.alaska.edu/GRD_MD/SA/S1A_EW_GRDM_1SDH_20230630T134749

In [15]:
# 1. 初始化GEE（只需运行一次）
if not initialize_gee():
    print("请先运行 ee.Authenticate() 进行认证")

# 2. 创建工作流程对象
workflow = VisualizationWorkflow(
    csv_file_path=r"E:\NWP\CS2_S1_matched\time_match_2023.csv",
    scene_name_column='sceneName',
    min_matched_points=60
)

# 3. 处理并可视化指定行（例如第0行）
row_data, match_result, visualization_map = workflow.process_and_visualize_row(
    row_index=0,
    roi=None,  # 可选：感兴趣区域
    zoom_start=10  # 地图缩放级别
)

# 4. 显示地图
if visualization_map:
    display(visualization_map)

# 5. 显示统计图表
if match_result and match_result['status'] == 'success':
    fig = workflow.visualizer.create_summary_plot(match_result)
    plt.show()

2025-09-02 15:28:29,638 - INFO - Earth Engine 初始化成功


✓ GEE 初始化成功!
✓ 成功加载CSV文件: 360 条记录
CSV列名: ['sceneName', 's1_start', 's1_end', 's1_url', 'cs2_path', 'cs2_time_min', 'cs2_time_max', 'num_matched_points']
筛选EW模式的S1数据...
应用额外筛选条件: num_matched_points > 60
✓ 筛选完成:
  原始数据: 360 条
  EW模式数据: 61 条
  最终筛选结果: 61 条

=== 第 0 行数据 ===
Scene Name: S1A_EW_GRDM_1SDH_20230630T134749_20230630T134853_049216_05EB02_5C83
s1_start: 2023-06-30 13:47:49
s1_end: 2023-06-30 13:48:53
s1_url: https://datapool.asf.alaska.edu/GRD_MD/SA/S1A_EW_GRDM_1SDH_20230630T134749_20230630T134853_049216_05EB02_5C83.zip
cs2_path: E:\NWP\CS2\2023\CS_OFFL_SIR_SIN_2__20230630T115732_20230630T120238_E001.nc
cs2_time_min: 2023-06-30 12:01:48.661628928
cs2_time_max: 2023-06-30 12:03:07.955597056
num_matched_points: 89

处理场景: S1A_EW_GRDM_1SDH_20230630T134749_20230630T134853_049216_05EB02_5C83
✓ 从scene name提取日期: 2023-06-30 13:47:49
搜索Sentinel-1 EW图像...
时间范围: 2023-06-27 到 2023-07-03


2025-09-02 15:28:30,116 - ERROR - 查找Sentinel-1图像失败: 'function' object has no attribute 'getFunction'


找到 366 个EW模式图像
处理失败: 'function' object has no attribute 'getFunction'
